In [1]:
# 1 — KAGGLE ENVIRONMENT + GLOBAL PATH REGISTRY (FINAL)

import os
import gc
import numpy as np
import pandas as pd

# --- FIND COMPETITION ROOT ---
BASE_CANDIDATES = [
    "/kaggle/input/competitions/stanford-rna-3d-folding-2",
    "/kaggle/input/stanford-rna-3d-folding-2",
]

COMP_ROOT = None
for p in BASE_CANDIDATES:
    if os.path.exists(p):
        COMP_ROOT = p
        break

if COMP_ROOT is None:
    raise FileNotFoundError(
        "❌ Competition dataset not found.\n"
        "👉 Click 'Add Input' and attach: Stanford RNA 3D Folding 2"
    )

# --- INPUT FILES ---
TRAIN_LABELS_PATH = os.path.join(COMP_ROOT, "train_labels.csv")
VALIDATION_LABELS_PATH = os.path.join(COMP_ROOT, "validation_labels.csv")
TRAIN_SEQS_PATH = os.path.join(COMP_ROOT, "train_sequences.csv")
VALIDATION_SEQS_PATH = os.path.join(COMP_ROOT, "validation_sequences.csv")
TEST_SEQS_PATH = os.path.join(COMP_ROOT, "test_sequences.csv")
SAMPLE_SUB_PATH = os.path.join(COMP_ROOT, "sample_submission.csv")

REQUIRED_FILES = [
    TRAIN_LABELS_PATH,
    VALIDATION_LABELS_PATH,
    TRAIN_SEQS_PATH,
    VALIDATION_SEQS_PATH,
    TEST_SEQS_PATH,
    SAMPLE_SUB_PATH,
]

missing_files = [p for p in REQUIRED_FILES if not os.path.exists(p)]
if missing_files:
    raise FileNotFoundError("❌ Missing required files:\n" + "\n".join(missing_files))

# --- WORKING OUTPUT PATHS ---
WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

FEATURE_OUT = os.path.join(WORK_DIR, "FEATURE_TABLE__GEOMETRY_LABELS_V1.csv")
BENCH_OUT   = os.path.join(WORK_DIR, "HELIX_GEOMETRY_RECON_BENCHMARK_V1.csv")

# --- PRINT STATE ---
print("✅ Competition root:", COMP_ROOT)
print("✅ Working dir:", WORK_DIR)
print("✅ Feature output:", FEATURE_OUT)
print("✅ Benchmark output:", BENCH_OUT)

print("\n📁 Files in competition root:")
for f in os.listdir(COMP_ROOT):
    print(" -", f)

✅ Competition root: /kaggle/input/competitions/stanford-rna-3d-folding-2
✅ Working dir: /kaggle/working
✅ Feature output: /kaggle/working/FEATURE_TABLE__GEOMETRY_LABELS_V1.csv
✅ Benchmark output: /kaggle/working/HELIX_GEOMETRY_RECON_BENCHMARK_V1.csv

📁 Files in competition root:
 - MSA
 - sample_submission.csv
 - validation_sequences.csv
 - test_sequences.csv
 - validation_labels.csv
 - extra
 - train_labels.csv
 - train_sequences.csv
 - PDB_RNA


In [2]:
# 2 — LOAD COMPETITION LABELS (MEMORY-SAFE, OPTIMIZED)

import pandas as pd
import numpy as np
import gc

USE_COLS = ["ID", "resname", "resid", "x_1", "y_1", "z_1", "chain", "copy"]

DTYPES = {
    "ID": "string",
    "resname": "category",
    "resid": "int32",
    "x_1": "float32",
    "y_1": "float32",
    "z_1": "float32",
    "chain": "category",
    "copy": "category",
}

print("📥 Loading train labels...")
df_labels = pd.read_csv(
    TRAIN_LABELS_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    low_memory=False
)

print("📥 Appending validation labels...")
df_val = pd.read_csv(
    VALIDATION_LABELS_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    low_memory=False
)

# Append WITHOUT creating large intermediate copies
df_labels = pd.concat([df_labels, df_val], ignore_index=True)
del df_val
gc.collect()

# --- DERIVE FIELDS ---
df_labels["target_id"] = df_labels["ID"].str.split("_").str[0]

df_labels["residue_index"] = df_labels["resid"].astype(np.int32)

df_labels["x"] = df_labels["x_1"]
df_labels["y"] = df_labels["y_1"]
df_labels["z"] = df_labels["z_1"]

# Drop original heavy columns EARLY
df_labels = df_labels.drop(columns=["ID", "resid", "x_1", "y_1", "z_1"])

gc.collect()

# --- SORT (critical for geometry) ---
df_labels = df_labels.sort_values(
    ["target_id", "chain", "copy", "residue_index"]
).reset_index(drop=True)

gc.collect()

# --- FINAL REPORT ---
print("✅ Label rows loaded:", len(df_labels))
print("✅ Targets:", df_labels["target_id"].nunique())
print("✅ Chains/copies:", df_labels.groupby(["target_id", "chain", "copy"]).ngroups)

print("\n📊 Memory usage (MB):", round(df_labels.memory_usage(deep=True).sum() / 1e6, 2))

df_labels.head(10)

📥 Loading train labels...
📥 Appending validation labels...
✅ Label rows loaded: 7804733
✅ Targets: 5744
✅ Chains/copies: 17952

📊 Memory usage (MB): 1330.14


,resname,chain,copy,target_id,residue_index,x,y,z
0,C,A,1,157D,1,4.843000,-5.640,13.265
1,G,A,1,157D,2,3.385000,-7.613,8.267
2,C,A,1,157D,3,2.158000,-6.751,2.949
3,G,A,1,157D,4,2.669000,-4.843,-1.773
4,A,A,1,157D,5,3.509000,0.239,-4.045
5,A,A,1,157D,6,6.073000,4.823,-5.035
6,U,A,1,157D,7,10.129000,7.706,-4.251
7,U,A,1,157D,8,15.514000,8.745,-2.055
8,A,A,1,157D,9,20.429001,7.281,-1.699
9,G,A,1,157D,10,23.509001,2.728,-1.918


In [3]:
# 3 — ENFORCE SEQUENTIAL GEOMETRY + PHYSICAL STEP FILTER

group_cols = ["target_id", "chain", "copy"]

df_geo = df_labels.copy()

df_geo["resid_diff"] = df_geo.groupby(group_cols)["residue_index"].diff()
df_geo["is_seq"] = df_geo["resid_diff"].eq(1)

df_geo = df_geo[df_geo["resid_diff"].isna() | df_geo["is_seq"]].copy()

df_geo["dx"] = df_geo.groupby(group_cols)["x"].diff()
df_geo["dy"] = df_geo.groupby(group_cols)["y"].diff()
df_geo["dz"] = df_geo.groupby(group_cols)["z"].diff()
df_geo["step"] = np.sqrt(df_geo["dx"]**2 + df_geo["dy"]**2 + df_geo["dz"]**2)

df_geo = df_geo[
    df_geo["step"].isna() | ((df_geo["step"] >= 4.5) & (df_geo["step"] <= 8.0))
].copy()

steps = df_geo["step"].dropna().values
print("✅ Sequential geometry rows:", len(df_geo))
print("✅ Step mean:", float(steps.mean()))
print("✅ Step median:", float(np.median(steps)))
print("✅ Step p90:", float(np.percentile(steps, 90)))
print("✅ Step min:", float(steps.min()))
print("✅ Step max:", float(steps.max()))

display(df_geo.head(10))


✅ Sequential geometry rows: 6996848
✅ Step mean: 5.791539669036865
✅ Step median: 5.574956893920898
✅ Step p90: 7.013174057006836
✅ Step min: 4.500007629394531
✅ Step max: 7.999997615814209


,resname,chain,copy,target_id,residue_index,x,y,z,resid_diff,is_seq,dx,dy,dz,step
0,C,A,1,157D,1,4.843000,-5.640,13.265,NaN,False,NaN,NaN,NaN,NaN
1,G,A,1,157D,2,3.385000,-7.613,8.267,1.0,True,-1.458000,-1.973,-4.998,5.567629
2,C,A,1,157D,3,2.158000,-6.751,2.949,1.0,True,-1.227000,0.862,-5.318,5.525369
3,G,A,1,157D,4,2.669000,-4.843,-1.773,1.0,True,0.511000,1.908,-4.722,5.118483
4,A,A,1,157D,5,3.509000,0.239,-4.045,1.0,True,0.840000,5.082,-2.272,5.629770
5,A,A,1,157D,6,6.073000,4.823,-5.035,1.0,True,2.564000,4.584,-0.990,5.344834
6,U,A,1,157D,7,10.129000,7.706,-4.251,1.0,True,4.056000,2.883,0.784,5.037606
7,U,A,1,157D,8,15.514000,8.745,-2.055,1.0,True,5.385000,1.039,2.196,5.907636
8,A,A,1,157D,9,20.429001,7.281,-1.699,1.0,True,4.915001,-1.464,0.356,5.140746
9,G,A,1,157D,10,23.509001,2.728,-1.918,1.0,True,3.080000,-4.553,-0.219,5.501288


In [4]:
# 4 — BUILD GEOMETRY FEATURES (NORMALIZED DIRECTION + CURVATURE + TURN ANGLE)

df_feat = df_geo.copy()

df_feat["step_safe"] = df_feat["step"].replace(0, np.nan)
df_feat["dx_norm"] = df_feat["dx"] / df_feat["step_safe"]
df_feat["dy_norm"] = df_feat["dy"] / df_feat["step_safe"]
df_feat["dz_norm"] = df_feat["dz"] / df_feat["step_safe"]

df_feat["dx_prev"] = df_feat.groupby(group_cols)["dx"].shift(1)
df_feat["dy_prev"] = df_feat.groupby(group_cols)["dy"].shift(1)
df_feat["dz_prev"] = df_feat.groupby(group_cols)["dz"].shift(1)
df_feat["step_prev"] = df_feat.groupby(group_cols)["step"].shift(1)

df_feat["dx_prev_norm"] = df_feat["dx_prev"] / df_feat["step_prev"]
df_feat["dy_prev_norm"] = df_feat["dy_prev"] / df_feat["step_prev"]
df_feat["dz_prev_norm"] = df_feat["dz_prev"] / df_feat["step_prev"]

df_feat["curvature"] = (
    df_feat["dx_norm"] * df_feat["dx_prev_norm"] +
    df_feat["dy_norm"] * df_feat["dy_prev_norm"] +
    df_feat["dz_norm"] * df_feat["dz_prev_norm"]
)
df_feat["curvature"] = np.clip(df_feat["curvature"], -1.0, 1.0)
df_feat["turn_angle"] = np.arccos(df_feat["curvature"])

feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

df_feat = df_feat.dropna(subset=feature_cols).reset_index(drop=True)
df_feat.to_csv(FEATURE_OUT, index=False)

print("✅ Feature table rows:", len(df_feat))
print("✅ Feature columns:", feature_cols)
print("✅ Saved:", FEATURE_OUT)

display(df_feat[feature_cols].describe())


✅ Feature table rows: 6441646
✅ Feature columns: ['dx_norm', 'dy_norm', 'dz_norm', 'curvature', 'turn_angle', 'step']
✅ Saved: /kaggle/working/FEATURE_TABLE__GEOMETRY_LABELS_V1.csv


,dx_norm,dy_norm,dz_norm,curvature,turn_angle,step
count,6.441646e+06,6.441646e+06,6.441646e+06,6.441646e+06,6.441646e+06,6.441646e+06
mean,-1.177979e-03,-4.055961e-04,5.236998e-05,6.556832e-01,7.731258e-01,5.791190e+00
std,5.763636e-01,5.767268e-01,5.749628e-01,4.014457e-01,4.991009e-01,7.265077e-01
min,-9.999998e-01,-1.000000e+00,-9.999996e-01,-9.999973e-01,0.000000e+00,4.500008e+00
25%,-5.014891e-01,-5.015000e-01,-4.989068e-01,6.167521e-01,4.604168e-01,5.306649e+00
50%,-3.073370e-03,-1.161220e-03,-1.881477e-04,8.187188e-01,6.116201e-01,5.574335e+00
75%,4.980182e-01,4.998140e-01,4.992287e-01,8.958674e-01,9.061865e-01,6.020129e+00
max,9.999999e-01,1.000000e+00,9.999996e-01,1.000000e+00,3.139277e+00,7.999998e+00


In [5]:
# 5 — LOAD GEOMETRY FEATURE TABLE (DATASET — FINAL, MEMORY SAFE)

import pandas as pd
import gc
import os

# --- DATASET PATH (CONFIRMED STRUCTURE) ---
FEATURE_PATH = "/kaggle/input/datasets/pharaohtut/helix-rna-geometry-v1/FEATURE_TABLE__GEOMETRY_FULL.parquet"

if not os.path.exists(FEATURE_PATH):
    raise FileNotFoundError(f"❌ Feature table not found at:\n{FEATURE_PATH}")

print("📥 Loading geometry feature table...")
print("Path:", FEATURE_PATH)

# --- LOAD PARQUET (COLUMN-OPTIMIZED) ---
USE_COLS = [
    "target_id",
    "chain",
    "copy",
    "residue_index",
    "x", "y", "z",
    "dx", "dy", "dz",
    "dx_norm", "dy_norm", "dz_norm",
    "curvature",
    "turn_angle",
    "step"
]

df_feat = pd.read_parquet(FEATURE_PATH, columns=USE_COLS)

# --- TYPE OPTIMIZATION (CRITICAL FOR MEMORY) ---
float_cols = [
    "x","y","z",
    "dx","dy","dz",
    "dx_norm","dy_norm","dz_norm",
    "curvature","turn_angle","step"
]

for col in float_cols:
    df_feat[col] = df_feat[col].astype("float32")

df_feat["residue_index"] = df_feat["residue_index"].astype("int32")

# --- SORT FOR SEQUENTIAL OPERATIONS ---
df_feat = df_feat.sort_values(
    ["target_id", "chain", "copy", "residue_index"]
).reset_index(drop=True)

# --- BASIC VALIDATION ---
print("\n✅ Feature table loaded")
print("Rows:", len(df_feat))
print("Targets:", df_feat["target_id"].nunique())
print("Chains:", df_feat.groupby(["target_id","chain","copy"]).ngroups)

print("\n📊 Sample:")
display(df_feat.head())

gc.collect()

📥 Loading geometry feature table...
Path: /kaggle/input/datasets/pharaohtut/helix-rna-geometry-v1/FEATURE_TABLE__GEOMETRY_FULL.parquet

✅ Feature table loaded
Rows: 7768829
Targets: 5744
Chains: 17746

📊 Sample:


,target_id,chain,copy,residue_index,x,y,z,dx,dy,dz,dx_norm,dy_norm,dz_norm,curvature,turn_angle,step
0,157D,A,1,2,3.385,-7.613,8.267,-1.458,-1.973,-4.998,-0.261871,-0.354370,-0.897689,0.521913,0.521913,5.567629
1,157D,A,1,3,2.158,-6.751,2.949,-1.227,0.862,-5.318,-0.222067,0.156008,-0.962470,0.392644,0.392644,5.525369
2,157D,A,1,4,2.669,-4.843,-1.773,0.511,1.908,-4.722,0.099834,0.372767,-0.922539,0.761646,0.761646,5.118483
3,157D,A,1,5,3.509,0.239,-4.045,0.840,5.082,-2.272,0.149207,0.902701,-0.403569,0.401361,0.401361,5.629770
4,157D,A,1,6,6.073,4.823,-5.035,2.564,4.584,-0.990,0.479716,0.857651,-0.185226,0.558137,0.558137,5.344834


0

In [6]:
# 11 — V09.1 FULL-SEND ENDGAME GENERATOR (HARDENED FINAL REPLACEMENT CELL)

import os
import math
import hashlib
import numpy as np
import pandas as pd

# ==============================================================================
# CONFIG
# ==============================================================================

STEP_LEN = 5.82
N_SAMPLES = 5
OUTPUT_PATH = "/kaggle/working/submission.csv"

COMP_ROOT_CANDIDATES = [
    "/kaggle/input/competitions/stanford-rna-3d-folding-2",
    "/kaggle/input/stanford-rna-3d-folding-2",
]
COMP_ROOT = None
for p in COMP_ROOT_CANDIDATES:
    if os.path.exists(p):
        COMP_ROOT = p
        break

if COMP_ROOT is None:
    raise FileNotFoundError(
        "❌ Competition root not found. Add the Stanford RNA 3D Folding 2 competition dataset."
    )

TEST_CSV = os.path.join(COMP_ROOT, "test_sequences.csv")
SAMPLE_SUB_CSV = os.path.join(COMP_ROOT, "sample_submission.csv")

REGIMES = [
    {
        "name": "compact",
        "memory": 0.82,
        "turn_amp": 0.48,
        "torsion_amp": 0.20,
        "compact_w": 0.18,
        "mid_w": 0.22,
        "repel_w": 0.09,
        "phase_amp": 0.18,
        "wander": 0.10,
        "anisotropy": 1.08,
    },
    {
        "name": "balanced",
        "memory": 0.78,
        "turn_amp": 0.40,
        "torsion_amp": 0.22,
        "compact_w": 0.12,
        "mid_w": 0.18,
        "repel_w": 0.08,
        "phase_amp": 0.22,
        "wander": 0.13,
        "anisotropy": 1.00,
    },
    {
        "name": "expanded",
        "memory": 0.74,
        "turn_amp": 0.30,
        "torsion_amp": 0.16,
        "compact_w": 0.05,
        "mid_w": 0.10,
        "repel_w": 0.10,
        "phase_amp": 0.28,
        "wander": 0.16,
        "anisotropy": 0.94,
    },
    {
        "name": "loop_biased",
        "memory": 0.80,
        "turn_amp": 0.58,
        "torsion_amp": 0.28,
        "compact_w": 0.14,
        "mid_w": 0.24,
        "repel_w": 0.08,
        "phase_amp": 0.30,
        "wander": 0.12,
        "anisotropy": 1.02,
    },
    {
        "name": "interaction_heavy",
        "memory": 0.83,
        "turn_amp": 0.42,
        "torsion_amp": 0.24,
        "compact_w": 0.20,
        "mid_w": 0.30,
        "repel_w": 0.11,
        "phase_amp": 0.20,
        "wander": 0.09,
        "anisotropy": 1.10,
    },
]

BASE_PRIOR = {
    "A": {"turn": 0.12, "torsion": 0.06},
    "C": {"turn": 0.22, "torsion": -0.02},
    "G": {"turn": 0.16, "torsion": 0.08},
    "U": {"turn": 0.28, "torsion": -0.05},
    "N": {"turn": 0.18, "torsion": 0.00},
}

# ==============================================================================
# UTILS
# ==============================================================================

def stable_hash_int(text: str, mod: int = 2**32) -> int:
    return int(hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:16], 16) % mod

def unit(v: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = float(np.linalg.norm(v))
    if n < eps:
        return np.array([1.0, 0.0, 0.0], dtype=np.float64)
    return v / n

def build_frame(direction: np.ndarray):
    t = unit(direction)
    ref = np.array([0.0, 0.0, 1.0], dtype=np.float64)
    if abs(np.dot(t, ref)) > 0.90:
        ref = np.array([0.0, 1.0, 0.0], dtype=np.float64)
    n = unit(np.cross(t, ref))
    b = unit(np.cross(t, n))
    return t, n, b

def rows_from_coords(target_id: str, seq: str, coords_stack: np.ndarray):
    rows = []
    L = len(seq)
    for i in range(L):
        row = {
            "ID": f"{target_id}_{i+1}",
            "resname": seq[i],
            "resid": i + 1,
        }
        for s in range(N_SAMPLES):
            x, y, z = coords_stack[s, i]
            row[f"x_{s+1}"] = float(x)
            row[f"y_{s+1}"] = float(y)
            row[f"z_{s+1}"] = float(z)
        rows.append(row)
    return rows

# ==============================================================================
# TRUSTED PRIOR EXTRACTION ONLY
# ==============================================================================

def normalize_reference_df(obj):
    if obj is None or not isinstance(obj, pd.DataFrame):
        return None
    if "ID" not in obj.columns:
        return None

    if {"x_1", "y_1", "z_1"}.issubset(obj.columns):
        xyz_cols = ("x_1", "y_1", "z_1")
    elif {"x", "y", "z"}.issubset(obj.columns):
        xyz_cols = ("x", "y", "z")
    else:
        return None

    ref = obj[["ID", *xyz_cols]].copy()
    ref["target_id"] = ref["ID"].astype(str).str.rsplit("_", n=1).str[0]
    ref["residue_index"] = pd.to_numeric(
        ref["ID"].astype(str).str.rsplit("_", n=1).str[-1],
        errors="coerce",
    )
    ref = ref.dropna(subset=["residue_index"]).copy()
    ref["residue_index"] = ref["residue_index"].astype(int)
    ref = ref.rename(columns={
        xyz_cols[0]: "x",
        xyz_cols[1]: "y",
        xyz_cols[2]: "z",
    })
    ref = ref.sort_values(["target_id", "residue_index"]).reset_index(drop=True)
    return ref

def select_reference_submission():
    # HARDENED: only trust explicitly named prior submission-style objects
    trusted_candidates = [
        "df_submission_fixed",
        "df_submission_prior",
        "df_submission_baseline",
        "submission_prior",
    ]

    for name in trusted_candidates:
        if name in globals():
            ref = normalize_reference_df(globals()[name])
            if ref is not None and len(ref) > 100:
                print(f"✅ Trusted motif prior source detected: {name}")
                return ref

    print("ℹ️ No trusted prior submission dataframe found. Using fallback motif priors only.")
    return None

def learn_motif_prior(reference_df: pd.DataFrame, test_df: pd.DataFrame):
    trigram_stats = {}
    base_stats = {k: {"turn": [], "torsion": []} for k in ["A", "C", "G", "U", "N"]}

    seq_map = dict(zip(test_df["target_id"], test_df["sequence"]))

    if reference_df is None:
        return {}, {k: BASE_PRIOR[k] for k in BASE_PRIOR}

    for tid, grp in reference_df.groupby("target_id", sort=False):
        if tid not in seq_map:
            continue

        seq = seq_map[tid]
        grp = grp.sort_values("residue_index").reset_index(drop=True)
        xyz = grp[["x", "y", "z"]].to_numpy(dtype=np.float64)

        L = min(len(seq), len(xyz))
        if L < 5:
            continue

        seq = seq[:L]
        xyz = xyz[:L]

        steps = xyz[1:] - xyz[:-1]
        step_norms = np.linalg.norm(steps, axis=1)
        good = step_norms > 1e-8

        if int(good.sum()) < 4:
            continue

        u = np.zeros_like(steps)
        u[good] = steps[good] / step_norms[good, None]

        for i in range(2, L - 1):
            if not good[i - 2] or not good[i - 1]:
                continue

            u_prev = u[i - 2]
            u_cur = u[i - 1]

            turn = float(np.clip(np.dot(u_prev, u_cur), -1.0, 1.0))
            cross = np.cross(u_prev, u_cur)
            torsion = float(cross[2] / (np.linalg.norm(cross) + 1e-8))

            tri = seq[i - 2:i + 1]
            center = seq[i - 1] if seq[i - 1] in "ACGU" else "N"

            trigram_stats.setdefault(tri, {"turn": [], "torsion": []})
            trigram_stats[tri]["turn"].append(turn)
            trigram_stats[tri]["torsion"].append(torsion)
            base_stats[center]["turn"].append(turn)
            base_stats[center]["torsion"].append(torsion)

    trigram_prior = {}
    for tri, vals in trigram_stats.items():
        trigram_prior[tri] = {
            "turn": float(np.mean(vals["turn"])),
            "torsion": float(np.mean(vals["torsion"])),
        }

    base_prior = {}
    for base, vals in base_stats.items():
        if len(vals["turn"]) == 0:
            base_prior[base] = BASE_PRIOR[base]
        else:
            base_prior[base] = {
                "turn": float(np.mean(vals["turn"])),
                "torsion": float(np.mean(vals["torsion"])),
            }

    print(f"✅ Learned motif priors: {len(trigram_prior)} trigram entries")
    return trigram_prior, base_prior

# ==============================================================================
# GENERATOR
# ==============================================================================

def get_prior(seq: str, i: int, trigram_prior: dict, base_prior: dict):
    tri = seq[max(0, i - 2):i + 1]
    if len(tri) == 3 and tri in trigram_prior:
        return trigram_prior[tri]
    base = seq[i - 1] if i - 1 >= 0 else "N"
    base = base if base in base_prior else "N"
    return base_prior.get(base, BASE_PRIOR["N"])

def init_seed_coords(seq: str, regime: dict, rng: np.random.Generator):
    L = len(seq)
    coords = np.zeros((L, 3), dtype=np.float64)

    theta1 = 0.60 + 0.18 * regime["turn_amp"] + rng.uniform(-0.08, 0.08)
    theta2 = 0.72 + 0.22 * regime["turn_amp"] + rng.uniform(-0.10, 0.10)
    phi = regime["torsion_amp"] * 0.65 + rng.uniform(-0.10, 0.10)

    coords[0] = np.array([0.0, 0.0, 0.0], dtype=np.float64)
    coords[1] = np.array([STEP_LEN, 0.0, 0.0], dtype=np.float64)

    d2 = unit(np.array([math.cos(theta1), math.sin(theta1), 0.20 * phi], dtype=np.float64))
    coords[2] = coords[1] + STEP_LEN * d2

    if L > 3:
        t, n, b = build_frame(coords[2] - coords[1])
        d3 = unit(
            0.82 * t
            + 0.50 * math.sin(theta2) * n
            + 0.35 * phi * b
        )
        coords[3] = coords[2] + STEP_LEN * d3

    return coords

def generate_track(
    target_id: str,
    seq: str,
    regime: dict,
    track_idx: int,
    trigram_prior: dict,
    base_prior: dict,
):
    L = len(seq)
    if L == 0:
        return np.zeros((0, 3), dtype=np.float64)
    if L == 1:
        return np.zeros((1, 3), dtype=np.float64)
    if L == 2:
        return np.array([[0.0, 0.0, 0.0], [STEP_LEN, 0.0, 0.0]], dtype=np.float64)

    rng = np.random.default_rng(stable_hash_int(f"{target_id}__{regime['name']}__{track_idx}"))
    coords = init_seed_coords(seq, regime, rng)

    phase = rng.uniform(0.0, 2.0 * math.pi)
    freq = rng.uniform(0.14, 0.30)
    z_bias = rng.uniform(-0.22, 0.22)
    side_sign = -1.0 if (track_idx % 2 == 0) else 1.0

    start_idx = 4 if L > 3 else 3

    for i in range(start_idx, L):
        prev = coords[i - 1]
        d1 = unit(coords[i - 1] - coords[i - 2])
        d0 = unit(coords[i - 2] - coords[i - 3]) if i >= 3 else d1
        t, n, b = build_frame(d1)

        prior = get_prior(seq, i, trigram_prior, base_prior)

        # Sequence-length-aware windows
        recent_window = max(8, min(24, L // 5))
        mid_window = max(12, min(40, L // 2))
        repel_window = max(10, min(28, L // 3))

        # 1) DIRECTIONAL MEMORY + MOTIF TURNING BIAS
        memory_vec = unit((1.0 + regime["memory"]) * d1 - regime["memory"] * d0)

        turn_strength = regime["turn_amp"] * (1.15 - 0.65 * prior["turn"])
        torsion_strength = regime["torsion_amp"] * (0.70 + 1.10 * abs(prior["torsion"]))

        motif_vec = (
            memory_vec
            + side_sign * turn_strength * n
            + np.sign(prior["torsion"] + 1e-8) * torsion_strength * b
            + z_bias * 0.10 * b
        )

        # 2) INTERACTION FIELD
        interaction = np.zeros(3, dtype=np.float64)

        # Recent compactness / spread bias
        hist_start = max(0, i - recent_window)
        recent = coords[hist_start:i]
        com_recent = recent.mean(axis=0)
        interaction += regime["compact_w"] * unit(com_recent - prev)

        # Mid-range return attraction
        mid_terms = []
        for j in range(max(0, i - mid_window), i - 6, 3):
            delta = coords[j] - prev
            dist = float(np.linalg.norm(delta))
            if dist < 1e-8:
                continue
            w = math.exp(-((dist - 15.0) ** 2) / (2.0 * 7.0 ** 2))
            mid_terms.append(w * unit(delta))
        if mid_terms:
            interaction += regime["mid_w"] * unit(np.sum(mid_terms, axis=0))

        # Weak long-ish pseudo-fold interaction
        if i > 18 and regime["name"] in {"compact", "interaction_heavy", "loop_biased"}:
            back_span = max(8, min(24, L // 4))
            anchor_idx = max(0, i - (back_span + (track_idx % 3) * max(2, back_span // 4)))
            interaction += 0.07 * unit(coords[anchor_idx] - prev)

        # Repulsion
        repulse = np.zeros(3, dtype=np.float64)
        for j in range(max(0, i - repel_window), i - 2):
            delta = prev - coords[j]
            dist = float(np.linalg.norm(delta))
            if 1e-8 < dist < 8.5:
                repulse += ((8.5 - dist) / 8.5) * unit(delta)
        interaction += regime["repel_w"] * repulse

        # 3) ENSEMBLE DECORRELATION PHASE FIELD
        phase_term = phase + freq * i
        phase_vec = (
            math.sin(phase_term) * n
            + regime["anisotropy"] * math.cos(phase_term * (1.17 + 0.04 * track_idx)) * b
        )
        interaction += regime["phase_amp"] * phase_vec

        # Bounded wander
        wander_vec = rng.normal(0.0, 1.0, size=3)
        wander_vec = (
            0.15 * wander_vec[0] * t
            + 1.00 * wander_vec[1] * n
            + 0.90 * wander_vec[2] * b
        )
        interaction += regime["wander"] * unit(wander_vec)

        # 4) COMBINE
        direction = unit(1.05 * motif_vec + interaction)

        # Anti-rod guard
        if i >= 6:
            recent_dirs = []
            for k in range(max(2, i - 5), i):
                recent_dirs.append(unit(coords[k] - coords[k - 1]))
            mean_align = float(np.mean([np.dot(direction, d) for d in recent_dirs]))
            if mean_align > 0.96:
                direction = unit(direction + 0.40 * n + 0.22 * b)

        coords[i] = prev + STEP_LEN * direction

    return coords

# ==============================================================================
# LOAD DATA
# ==============================================================================

if "df_test" in globals() and isinstance(globals()["df_test"], pd.DataFrame) and {"target_id", "sequence"}.issubset(globals()["df_test"].columns):
    test_df = globals()["df_test"][["target_id", "sequence"]].copy().reset_index(drop=True)
else:
    test_df = pd.read_csv(TEST_CSV)

sample_sub = pd.read_csv(SAMPLE_SUB_CSV)

print(f"✅ Test targets loaded: {len(test_df):,}")

reference_df = select_reference_submission()
TRIGRAM_PRIOR, LEARNED_BASE_PRIOR = learn_motif_prior(reference_df, test_df)

FINAL_BASE_PRIOR = BASE_PRIOR.copy()
for k, v in LEARNED_BASE_PRIOR.items():
    FINAL_BASE_PRIOR[k] = v

# ==============================================================================
# GENERATE ALL TRACKS
# ==============================================================================

all_rows = []

for row in test_df.itertuples(index=False):
    target_id = row.target_id
    seq = row.sequence

    coords_tracks = []
    for track_idx, regime in enumerate(REGIMES):
        coords = generate_track(
            target_id=target_id,
            seq=seq,
            regime=regime,
            track_idx=track_idx,
            trigram_prior=TRIGRAM_PRIOR,
            base_prior=FINAL_BASE_PRIOR,
        )
        coords_tracks.append(coords)

    coords_stack = np.stack(coords_tracks, axis=0)
    all_rows.extend(rows_from_coords(target_id, seq, coords_stack))

submission = pd.DataFrame(all_rows)

# ==============================================================================
# FINAL SAFETY / SCHEMA
# ==============================================================================

expected_cols = ["ID", "resname", "resid"] + [
    f"{axis}_{i}" for i in range(1, N_SAMPLES + 1) for axis in ["x", "y", "z"]
]

for c in expected_cols:
    if c not in submission.columns:
        submission[c] = 0.0 if c not in {"ID", "resname"} else ""

submission = submission[expected_cols].copy()

coord_cols = [c for c in submission.columns if c.startswith(("x_", "y_", "z_"))]
submission[coord_cols] = submission[coord_cols].replace([np.inf, -np.inf], 0.0).fillna(0.0)
submission[coord_cols] = submission[coord_cols].clip(-5000.0, 5000.0)

# HARD FAIL coverage check
merged = sample_sub[["ID", "resname", "resid"]].merge(
    submission,
    on=["ID", "resname", "resid"],
    how="left",
    sort=False,
    indicator=True,
)

missing_rows = int((merged["_merge"] != "both").sum())
if missing_rows > 0:
    bad_rows = merged.loc[merged["_merge"] != "both", ["ID", "resname", "resid"]].head(10)
    raise RuntimeError(
        f"❌ Submission merge coverage failure: {missing_rows} rows did not match sample submission.\n"
        f"First mismatches:\n{bad_rows}"
    )

submission = merged.drop(columns=["_merge"]).copy()

if list(submission.columns) != list(sample_sub.columns):
    raise RuntimeError(
        "❌ Final submission columns do not exactly match sample_submission columns."
    )

submission[coord_cols] = submission[coord_cols].replace([np.inf, -np.inf], 0.0).fillna(0.0)
submission[coord_cols] = submission[coord_cols].astype(np.float32)

# ==============================================================================
# REGIME AUDIT
# ==============================================================================

def radius_of_gyration(coords: np.ndarray) -> float:
    center = coords.mean(axis=0, keepdims=True)
    return float(np.sqrt(np.mean(np.sum((coords - center) ** 2, axis=1))))

regime_stats = []

for s in range(N_SAMPLES):
    xcol, ycol, zcol = f"x_{s+1}", f"y_{s+1}", f"z_{s+1}"
    tmp = submission[["ID", xcol, ycol, zcol]].copy()
    tmp["target_id"] = tmp["ID"].astype(str).str.rsplit("_", n=1).str[0]
    tmp["residue_index"] = pd.to_numeric(
        tmp["ID"].astype(str).str.rsplit("_", n=1).str[-1],
        errors="coerce",
    )
    tmp = tmp.dropna(subset=["residue_index"]).copy()
    tmp["residue_index"] = tmp["residue_index"].astype(int)

    step_vals = []
    rog_vals = []

    for tid, grp in tmp.groupby("target_id", sort=False):
        grp = grp.sort_values("residue_index")
        xyz = grp[[xcol, ycol, zcol]].to_numpy(dtype=np.float64)
        if len(xyz) >= 2:
            steps = np.linalg.norm(xyz[1:] - xyz[:-1], axis=1)
            step_vals.append(float(np.mean(steps)))
        if len(xyz) >= 3:
            rog_vals.append(radius_of_gyration(xyz))

    regime_stats.append({
        "sample": s + 1,
        "regime": REGIMES[s]["name"],
        "mean_step": float(np.mean(step_vals)) if step_vals else np.nan,
        "mean_rog": float(np.mean(rog_vals)) if rog_vals else np.nan,
    })

df_regime_stats = pd.DataFrame(regime_stats)

# ==============================================================================
# SAVE + AUDIT
# ==============================================================================

submission.to_csv(OUTPUT_PATH, index=False)

vals = submission[coord_cols].to_numpy(dtype=np.float64)
nan_count = int(np.isnan(vals).sum())

sub1 = submission[["ID", "x_1", "y_1", "z_1"]].copy()
sub1["target_id"] = sub1["ID"].astype(str).str.rsplit("_", n=1).str[0]
sub1["residue_index"] = pd.to_numeric(
    sub1["ID"].astype(str).str.rsplit("_", n=1).str[-1],
    errors="coerce",
)
sub1 = sub1.dropna(subset=["residue_index"]).copy()
sub1["residue_index"] = sub1["residue_index"].astype(int)
sub1 = sub1.sort_values(["target_id", "residue_index"]).reset_index(drop=True)

sub1["x_next"] = sub1.groupby("target_id")["x_1"].shift(-1)
sub1["y_next"] = sub1.groupby("target_id")["y_1"].shift(-1)
sub1["z_next"] = sub1.groupby("target_id")["z_1"].shift(-1)
sub1["step_len"] = np.sqrt(
    (sub1["x_next"] - sub1["x_1"]) ** 2
    + (sub1["y_next"] - sub1["y_1"]) ** 2
    + (sub1["z_next"] - sub1["z_1"]) ** 2
)
step_stats = sub1["step_len"].dropna().describe()

print("\n" + "=" * 80)
print("✅ V09.1 FULL-SEND GENERATOR COMPLETE")
print("=" * 80)
print(f"Output: {OUTPUT_PATH}")
print(f"Rows: {len(submission):,}")
print(f"NaN count: {nan_count}")

print("\n📐 STEP LENGTH STATS (sample 1):")
print(step_stats)

print("\n🧬 REGIME AUDIT:")
print(df_regime_stats)

print("\n🧬 First rows:")
print(submission.head(5))

✅ Test targets loaded: 28
ℹ️ No trusted prior submission dataframe found. Using fallback motif priors only.

✅ V09.1 FULL-SEND GENERATOR COMPLETE
Output: /kaggle/working/submission.csv
Rows: 9,762
NaN count: 0

📐 STEP LENGTH STATS (sample 1):
count    9734.000000
mean        5.820000
std         0.000256
min         5.819998
25%         5.820000
50%         5.820000
75%         5.820000
max         5.820002
Name: step_len, dtype: float64

🧬 REGIME AUDIT:
   sample             regime  mean_step   mean_rog
0       1            compact       5.82   8.001242
1       2           balanced       5.82  11.722662
2       3           expanded       5.82  29.880657
3       4        loop_biased       5.82   8.763433
4       5  interaction_heavy       5.82   9.215207

🧬 First rows:
       ID resname  resid        x_1       y_1       z_1       x_2       y_2  \
0  8ZNQ_1       A      1   0.000000  0.000000  0.000000   0.00000  0.000000   
1  8ZNQ_2       C      2   5.820000  0.000000  0.000000   5.82

In [7]:
# 12 — VALIDATE AND SAVE FINAL SUBMISSION (STRICT — SCHEMA LOCKED)

import numpy as np
import pandas as pd

OUT_PATH = "/kaggle/working/submission.csv"

print("🔍 Running strict submission validation...")

sample = pd.read_csv(SAMPLE_SUB_PATH)

# --- HARD SCHEMA LOCK ---
expected_cols = list(sample.columns)

missing_cols = [c for c in expected_cols if c not in submission.columns]
if missing_cols:
    raise ValueError(f"❌ Submission is missing required columns: {missing_cols}")

extra_cols = [c for c in submission.columns if c not in expected_cols]
if extra_cols:
    print(f"⚠️ Dropping extra columns not allowed by sample_submission: {extra_cols}")

# Keep only exact sample columns and exact order
submission = submission.loc[:, expected_cols].copy()

# --- BASIC STRUCTURE CHECKS ---
if list(submission.columns) != expected_cols:
    raise ValueError(
        "❌ Column mismatch after schema lock\n\n"
        f"Expected:\n{expected_cols}\n\n"
        f"Got:\n{list(submission.columns)}"
    )

if len(submission) != len(sample):
    raise ValueError(
        f"❌ Row count mismatch: expected {len(sample)}, got {len(submission)}"
    )

# --- ID VALIDATION ---
sub_ids = submission["ID"].astype(str).str.strip().reset_index(drop=True)
sample_ids = sample["ID"].astype(str).str.strip().reset_index(drop=True)

if not sub_ids.equals(sample_ids):
    mismatches = (sub_ids != sample_ids)
    mismatch_indices = np.where(mismatches)[0][:10]

    debug_rows = pd.DataFrame({
        "submission_ID": sub_ids.iloc[mismatch_indices].tolist(),
        "sample_ID": sample_ids.iloc[mismatch_indices].tolist(),
    })

    raise ValueError(
        "❌ Submission IDs do NOT match sample_submission.csv\n\n"
        f"First mismatches:\n{debug_rows}"
    )

print("✅ ID alignment PASSED")

# --- COORDINATE VALIDATION ---
coord_cols = [c for c in submission.columns if c.startswith(("x_", "y_", "z_"))]

if len(coord_cols) == 0:
    raise ValueError("❌ No coordinate columns found (x_*, y_*, z_*)")

# Force numeric before validation
for c in coord_cols:
    submission[c] = pd.to_numeric(submission[c], errors="raise")

if submission[coord_cols].isnull().any().any():
    bad = submission[coord_cols].isnull().sum()
    raise ValueError(f"❌ Null values detected:\n{bad[bad > 0]}")

vals = submission[coord_cols].to_numpy(dtype=np.float64)
if not np.isfinite(vals).all():
    raise ValueError("❌ Non-finite values found (NaN or Inf)")

print("✅ Coordinate validity PASSED")

# --- TYPE ENFORCEMENT ---
submission["resname"] = submission["resname"].astype(str)
submission["resid"] = pd.to_numeric(submission["resid"], errors="raise").astype(np.int32)
for c in coord_cols:
    submission[c] = submission[c].astype(np.float32)

print("✅ Dtype enforcement PASSED")

# --- FINAL SAVE ---
submission.to_csv(OUT_PATH, index=False)

print("\n🚀 SUBMISSION READY")
print("📁 Saved to:", OUT_PATH)
print("📐 Shape:", submission.shape)
print("📊 Coordinate dtype sample:")
print(submission[coord_cols].dtypes.head())

🔍 Running strict submission validation...
✅ ID alignment PASSED
✅ Coordinate validity PASSED
✅ Dtype enforcement PASSED

🚀 SUBMISSION READY
📁 Saved to: /kaggle/working/submission.csv
📐 Shape: (9762, 18)
📊 Coordinate dtype sample:
x_1    float32
y_1    float32
z_1    float32
x_2    float32
y_2    float32
dtype: object


In [8]:
# DIAG_BRIDGE_UNIVERSAL — DATASET-AGNOSTIC BRIDGE OUTPUT ENGINE

import os
import json
import numpy as np
import pandas as pd

OUTPUT_ROOT = "./bridge_outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

GEOMETRY_OUT = os.path.join(OUTPUT_ROOT, "BRIDGE_GEOMETRY_UNITS.csv")
METHOD_OUT   = os.path.join(OUTPUT_ROOT, "BRIDGE_METHOD_OUTPUTS.csv")
REALITY_OUT  = os.path.join(OUTPUT_ROOT, "BRIDGE_STRUCTURAL_REALITY.csv")
MANIFEST_OUT = os.path.join(OUTPUT_ROOT, "GENERIC_RUN_MANIFEST.json")

# ============================================================
# 1 — INPUT DETECTION (FLEXIBLE)
# ============================================================

df_input = None

CANDIDATES = ["submission", "df_submission", "df_coords", "df"]

for name in CANDIDATES:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        df_input = globals()[name].copy()
        print(f"✅ Using input dataframe: {name}")
        break

if df_input is None:
    raise RuntimeError("❌ No valid dataframe found in globals()")

# ============================================================
# 2 — COLUMN NORMALIZATION
# ============================================================

if "ID" in df_input.columns:
    df_input["target_id"] = df_input["ID"].astype(str).str.rsplit("_", n=1).str[0]
    df_input["residue_index"] = pd.to_numeric(
        df_input["ID"].astype(str).str.rsplit("_", n=1).str[-1],
        errors="coerce"
    )
else:
    raise RuntimeError("❌ ID column required")

df_input = df_input.dropna(subset=["residue_index"]).copy()
df_input["residue_index"] = df_input["residue_index"].astype(int)

# detect coordinate sets
coord_sets = []
for i in range(1, 6):
    if f"x_{i}" in df_input.columns:
        coord_sets.append((f"x_{i}", f"y_{i}", f"z_{i}"))

if len(coord_sets) == 0:
    if {"x","y","z"}.issubset(df_input.columns):
        coord_sets = [("x","y","z")]
    else:
        raise RuntimeError("❌ No coordinate columns found")

print(f"✅ Detected {len(coord_sets)} coordinate tracks")

# ============================================================
# 3 — GEOMETRY UNITS
# ============================================================

geometry_rows = []

for (xcol, ycol, zcol) in coord_sets:
    for tid, grp in df_input.groupby("target_id"):
        grp = grp.sort_values("residue_index")
        xyz = grp[[xcol, ycol, zcol]].values.astype(np.float64)

        if len(xyz) < 2:
            continue

        deltas = xyz[1:] - xyz[:-1]
        steps = np.linalg.norm(deltas, axis=1)

        for i in range(len(steps)):
            geometry_rows.append({
                "target_id": tid,
                "residue_index": int(grp["residue_index"].iloc[i]),
                "step": float(steps[i]),
                "dx": float(deltas[i][0]),
                "dy": float(deltas[i][1]),
                "dz": float(deltas[i][2]),
                "track": xcol
            })

df_geometry = pd.DataFrame(geometry_rows)
df_geometry.to_csv(GEOMETRY_OUT, index=False)

print("✅ Geometry units written:", GEOMETRY_OUT)

# ============================================================
# 4 — METHOD OUTPUTS (PER TRACK SUMMARY)
# ============================================================

method_rows = []

for (xcol, ycol, zcol) in coord_sets:
    steps_all = []
    rog_all = []

    for tid, grp in df_input.groupby("target_id"):
        grp = grp.sort_values("residue_index")
        xyz = grp[[xcol, ycol, zcol]].values.astype(np.float64)

        if len(xyz) < 2:
            continue

        steps = np.linalg.norm(xyz[1:] - xyz[:-1], axis=1)
        steps_all.extend(steps)

        center = xyz.mean(axis=0)
        rog = np.sqrt(np.mean(np.sum((xyz - center)**2, axis=1)))
        rog_all.append(rog)

    method_rows.append({
        "track": xcol,
        "mean_step": float(np.mean(steps_all)),
        "std_step": float(np.std(steps_all)),
        "mean_rog": float(np.mean(rog_all)),
    })

df_method = pd.DataFrame(method_rows)
df_method.to_csv(METHOD_OUT, index=False)

print("✅ Method outputs written:", METHOD_OUT)

# ============================================================
# 5 — STRUCTURAL REALITY
# ============================================================

reality_rows = []

for (xcol, ycol, zcol) in coord_sets:
    for tid, grp in df_input.groupby("target_id"):
        grp = grp.sort_values("residue_index")
        xyz = grp[[xcol, ycol, zcol]].values.astype(np.float64)

        if len(xyz) < 3:
            continue

        steps = np.linalg.norm(xyz[1:] - xyz[:-1], axis=1)
        center = xyz.mean(axis=0)
        rog = np.sqrt(np.mean(np.sum((xyz - center)**2, axis=1)))

        status = "VALID"

        if np.std(steps) > 1.5:
            status = "STEP_UNSTABLE"
        elif rog < 5:
            status = "COLLAPSED"
        elif rog > 300:
            status = "EXPLODED"

        reality_rows.append({
            "target_id": tid,
            "track": xcol,
            "mean_step": float(np.mean(steps)),
            "std_step": float(np.std(steps)),
            "radius_of_gyration": float(rog),
            "status": status
        })

df_reality = pd.DataFrame(reality_rows)
df_reality.to_csv(REALITY_OUT, index=False)

print("✅ Structural reality written:", REALITY_OUT)

# ============================================================
# 6 — MANIFEST
# ============================================================

manifest = {
    "rows_input": int(len(df_input)),
    "targets": int(df_input["target_id"].nunique()),
    "tracks": len(coord_sets),
    "geometry_rows": int(len(df_geometry)),
    "status": "COMPLETE"
}

with open(MANIFEST_OUT, "w") as f:
    json.dump(manifest, f, indent=2)

print("✅ Manifest written:", MANIFEST_OUT)

print("\n" + "="*60)
print("🚀 UNIVERSAL BRIDGE COMPLETE")
print("="*60)

✅ Using input dataframe: submission
✅ Detected 5 coordinate tracks
✅ Geometry units written: ./bridge_outputs/BRIDGE_GEOMETRY_UNITS.csv
✅ Method outputs written: ./bridge_outputs/BRIDGE_METHOD_OUTPUTS.csv
✅ Structural reality written: ./bridge_outputs/BRIDGE_STRUCTURAL_REALITY.csv
✅ Manifest written: ./bridge_outputs/GENERIC_RUN_MANIFEST.json

🚀 UNIVERSAL BRIDGE COMPLETE
